In [1]:
#Task 1: GridSearchCV on RandomForest for review sentiment
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

rv = pd.read_csv("flipkart_reviews.csv")
X = rv["review"]
y = rv["label"]
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

pipe = make_pipeline(CountVectorizer(stop_words="english"), RandomForestClassifier(random_state=1))
grid = {
    "randomforestclassifier__max_depth": [3, 5, 10, None],
    "randomforestclassifier__n_estimators": [20, 50, 100],
}

gs = GridSearchCV(pipe, grid, cv=5)
gs.fit(xtr, ytr)

print("Best parameters:", gs.best_params_)
print("Best cross-validation accuracy:", round(gs.best_score_, 3))
print("Test accuracy:", round(gs.score(xte, yte), 3))

Best parameters: {'randomforestclassifier__max_depth': None, 'randomforestclassifier__n_estimators': 20}
Best cross-validation accuracy: 0.933
Test accuracy: 0.933


In [2]:
#Task 2: RandomizedSearchCV on SVM for Instagram engagement
from scipy.stats import loguniform
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

ig = pd.read_csv("instagram_engagement.csv")
Xi = ig[["caption_length", "posting_hour", "hashtag_count"]]
yi = ig["high_engagement"]
xtr_i, xte_i, ytr_i, yte_i = train_test_split(Xi, yi, test_size=0.25, random_state=1, stratify=yi)

svm_pipe = make_pipeline(StandardScaler(), SVC())
dist = {
    "svc__C": loguniform(0.1, 100),
    "svc__gamma": loguniform(0.001, 1),
}

rs = RandomizedSearchCV(svm_pipe, dist, n_iter=10, cv=5, random_state=1)
rs.fit(xtr_i, ytr_i)

print("Best parameters:", {k: round(float(v), 3) for k, v in rs.best_params_.items()})
print("Best cross-validation accuracy:", round(rs.best_score_, 3))
print("Test accuracy:", round(rs.score(xte_i, yte_i), 3))

Best parameters: {'svc__C': 1.783, 'svc__gamma': 0.145}
Best cross-validation accuracy: 0.744
Test accuracy: 0.76


In [3]:
#Task 3: SMOTE on an imbalanced dataset and GridSearchCV on RandomForest
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import classification_report

df = pd.read_csv("creditcard.csv")
Xc = df.drop(columns="Class")
yc = df["Class"]
xtr_c, xte_c, ytr_c, yte_c = train_test_split(Xc, yc, test_size=0.3, random_state=1, stratify=yc)

print("Class counts before SMOTE:", {int(k): int(v) for k, v in ytr_c.value_counts().items()})
xs, ys = SMOTE(random_state=1).fit_resample(xtr_c, ytr_c)
print("Class counts after SMOTE:", {int(k): int(v) for k, v in ys.value_counts().items()})

# SMOTE is inside the pipeline, so it is applied only on the training part of each fold (no leakage into the validation fold)
sm_pipe = ImbPipeline([("smote", SMOTE(random_state=1)), ("rf", RandomForestClassifier(random_state=1))])
grid3 = {"rf__max_depth": [5, 10, None], "rf__n_estimators": [50, 100]}

gs3 = GridSearchCV(sm_pipe, grid3, cv=3, scoring="f1")
gs3.fit(xtr_c, ytr_c)

print("\nBest parameters:", gs3.best_params_)
print("Best cross-validation F1:", round(gs3.best_score_, 3))
print("\nTest set report with SMOTE + tuned RandomForest:")
print(classification_report(yte_c, gs3.predict(xte_c)))

# SMOTE makes both classes equal in the training data (3165 each). The tuned model finds about 83% of the frauds (recall)
# and about 72% of its fraud alerts are correct (precision).

Class counts before SMOTE: {0: 3165, 1: 335}
Class counts after SMOTE: {0: 3165, 1: 3165}



Best parameters: {'rf__max_depth': 10, 'rf__n_estimators': 100}
Best cross-validation F1: 0.814

Test set report with SMOTE + tuned RandomForest:
              precision    recall  f1-score   support

           0       0.98      0.97      0.97      1357
           1       0.72      0.83      0.77       143

    accuracy                           0.95      1500
   macro avg       0.85      0.90      0.87      1500
weighted avg       0.96      0.95      0.95      1500



In [4]:
#Task 4: class_weight='balanced' with SVM tuning (accuracy and F1 before and after)
svm_grid = {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.01, 0.1]}

for cw in [None, "balanced"]:
    p = make_pipeline(StandardScaler(), SVC(class_weight=cw))
    g = GridSearchCV(p, svm_grid, cv=3, scoring="f1")
    g.fit(xtr_c, ytr_c)
    pred = g.predict(xte_c)
    print("class_weight =", cw)
    print("  best parameters:", g.best_params_)
    print("  accuracy:", round(accuracy_score(yte_c, pred), 3))
    print("  F1 (fraud class):", round(f1_score(yte_c, pred), 3))
    print("  recall (fraud class):", round((pred[yte_c.values == 1] == 1).mean(), 3))

# With class_weight='balanced' the recall of the fraud class goes up (0.804 to 0.874), but accuracy (0.97 to 0.947) and F1 (0.836 to 0.76) go down.
# The model now flags more normal transactions as fraud, so precision falls. Balanced weights help to find more of the minority class
# but they cost some precision. If missing a fraud is worse than a false alarm, balanced is the better choice.

class_weight = None
  best parameters: {'svc__C': 10, 'svc__gamma': 0.01}
  accuracy: 0.97
  F1 (fraud class): 0.836
  recall (fraud class): 0.804


class_weight = balanced
  best parameters: {'svc__C': 1, 'svc__gamma': 'scale'}
  accuracy: 0.947
  F1 (fraud class): 0.76
  recall (fraud class): 0.874


In [5]:
#Task 5: Parameter grid for RandomForest on the Wine dataset
from sklearn.datasets import load_wine

wine = load_wine()
xw_tr, xw_te, yw_tr, yw_te = train_test_split(wine.data, wine.target, test_size=0.25, random_state=1, stratify=wine.target)

grid5 = {
    "n_estimators": [50, 100, 200],     # more trees give a more stable vote, but after about 200 the gain is small and it gets slower
    "max_depth": [None, 5, 10],         # None lets trees grow fully, small depths limit overfitting
    "min_samples_split": [2, 5],        # a bigger value stops the trees from splitting on very few samples
    "max_features": ["sqrt", "log2"],   # fewer features per split makes the trees more different from each other
}

gs5 = GridSearchCV(RandomForestClassifier(random_state=1), grid5, cv=5)
gs5.fit(xw_tr, yw_tr)

print("Number of combinations tried:", len(gs5.cv_results_["params"]))
print("Best parameters:", gs5.best_params_)
print("Best cross-validation accuracy:", round(gs5.best_score_, 3))
print("Test accuracy:", round(gs5.score(xw_te, yw_te), 3))

Number of combinations tried: 36
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 50}
Best cross-validation accuracy: 0.977
Test accuracy: 1.0
